## Gold Layer

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
df_silver = spark.read.table("fintech.silver.stock_prices")

### Update fact_stock_prices

In [0]:
# =========================================================
# Read Gold dimensions
# =========================================================
df_dim_stock = spark.table(
    "fintech.gold.dim_stock"
)

df_dim_date = spark.table(
    "fintech.gold.dim_date"
)

# =========================================================
# Build fact dataset
# =========================================================
df_fact_stock_prices = (
    df_silver.alias("p")

    .join(
        df_dim_stock.alias("s"),
        on= F.col("p.symbol") == F.col("s.symbol"),
        how="inner"
    )

    .join(
        df_dim_date.alias("d"),
        on = F.col("p.trade_date") == F.col("d.full_date"),
        how="inner"
    )

    .select(
        F.col("s.stock_key"),
        F.col("d.date_key"),

        F.col("p.open"),
        F.col("p.high"),
        F.col("p.low"),
        F.col("p.close"),
        F.col("p.volume"),
        F.col("p.change"),
        F.col("p.change_percent"),
        F.col("p.vwap"),

        F.col("p._ingestion_timestamp"),
        F.col("p._source_file")

    )
)

# =========================================================
# Check if fact table exists
# =========================================================
fact_exists = spark.catalog.tableExists(
    "fintech.gold.fact_stock_prices"
)

# =========================================================
# Create fact table if it does not exist
# =========================================================
if not fact_exists:

    (
        df_fact_stock_prices
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("fintech.gold.fact_stock_prices")
    )

# =========================================================
# Incremental MERGE
# =========================================================
else:
    fact_table = DeltaTable.forName(
        spark, "fintech.gold.fact_stock_prices"
    )

    (
        fact_table.alias("target")
        .merge(
            df_fact_stock_prices.alias("source"),
            """
                target.stock_key = source.stock_key
                AND target.date_key = source.date_key
            """
        )
        .whenMatchedUpdate(
            condition="""
                source._ingestion_timestamp > target._ingestion_timestamp
            """,
            set={
                "open": "source.open",
                "high": "source.high",
                "low": "source.low",
                "close": "source.close",
                "volume": "source.volume",
                "change": "source.change",
                "change_percent": "source.change_percent",
                "vwap": "source.vwap",
                "_ingestion_timestamp":
                    "source._ingestion_timestamp",
                "_source_file":
                    "source._source_file"
            }
        )

        .whenNotMatchedInsertAll()
        
        .execute()
    )